# Mel Spectrograms and MFCCs

Squeeze a spectrogram through a mel filterbank, take the log, apply a DCT, and check each step against librosa.

In [ ]:
import sys
sys.path.insert(0, '../code')
import numpy as np
from main import hz_to_mel, mel_filterbank, mel_spectrogram, mfcc, power_to_db, synthetic_vowel

sample_rate, n_fft, hop_length, n_mels = 16_000, 512, 160, 40
filters = mel_filterbank(sample_rate, n_fft, n_mels)
print('filterbank', filters.shape)
print('mel of 500, 1000, 2000, 4000, 8000 Hz:', np.round(hz_to_mel([500, 1_000, 2_000, 4_000, 8_000])))

In [ ]:
vowel_a = (730.0, 1_090.0, 2_440.0)
vowel_i = (270.0, 2_290.0, 3_010.0)

def profile(formants, pitch):
    return mfcc(synthetic_vowel(formants, pitch, sample_rate, 0.5), sample_rate)[1:].mean(axis=1)

same_vowel = np.linalg.norm(profile(vowel_a, 120.0) - profile(vowel_a, 200.0))
same_pitch = np.linalg.norm(profile(vowel_a, 120.0) - profile(vowel_i, 120.0))
round(float(same_vowel), 1), round(float(same_pitch), 1)

MFCCs describe the spectral envelope, so changing the pitch barely moves them while changing the vowel moves them a lot. Try pitches of 90 Hz and 300 Hz. At which pitch does the "same vowel" distance start to grow, and why do the low mel bands begin to resolve single harmonics?

In [ ]:
import librosa

y = synthetic_vowel(vowel_a, 120.0, sample_rate, 0.5)
ours = mel_spectrogram(y, sample_rate, n_fft, hop_length, n_mels)
theirs = librosa.feature.melspectrogram(y=y, sr=sample_rate, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels, htk=True)
print(f'mel spectrogram, relative max difference: {np.max(np.abs(ours - theirs)) / theirs.max():.1e}')

ours_mfcc = mfcc(y, sample_rate, 13, n_fft, hop_length, n_mels)
theirs_mfcc = librosa.feature.mfcc(S=librosa.power_to_db(theirs), n_mfcc=13)
print(f'MFCC, max difference: {np.max(np.abs(ours_mfcc - theirs_mfcc)):.1e}')

The remaining differences come from librosa storing its filterbank in float32. Now drop `htk=True` from the librosa call. The default Slaney mel scale is linear below 1 kHz, so the band edges move and the numbers no longer match. Which one does your downstream model expect?